In [11]:
# Import necessary libraries
import os
from dotenv import load_dotenv
from groq import Groq
import json

# Load environment variables from .env file (e.g., GROQ_API_KEY)
load_dotenv()

# Initialize the Groq client and specify the model to use
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
model = "llama-3.3-70b-versatile"

In [12]:
def chat(messages, system=None, temperature=0.0, stop_sequences=[]):
    if system:
        messages = [{"role": "system", "content": system}] + messages

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop": stop_sequences if stop_sequences else None,
    }

    completion = client.chat.completions.create(**params)
    return completion.choices[0].message.content

In [ ]:
# Function to generate an evaluation dataset using the LLM
def generate_dataset():
    """Instructs the LLM to generate a set of AWS-related coding/config tasks for evaluation"""
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages=[]
    add_user_message(messages, prompt)
    # Pre-fill with ```json to encourage JSON output
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [15]:
# Generate the dataset and save it to a local JSON file
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [17]:
# Function to format a test case and get the model's output
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


In [18]:
# Function to execute a test case and assign a score
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # Placeholder for grading logic (currently returns a fixed score)
    # TODO - Grading
    score = 10

    return {"output": output, "test_case": test_case, "score": score}


In [19]:
# Function to iterate through the entire dataset and collect results
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results


In [21]:
# Load the dataset from file, execute the evaluation, and print the resulting JSON
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))


[
  {
    "output": "### Extract Region from AWS EC2 Instance Metadata\n#### Function Description\nThis function takes an AWS EC2 instance metadata JSON response as input and returns the 'Region' as a string.\n\n#### Code\n```python\nimport json\n\ndef extract_region(metadata_json):\n    \"\"\"\n    Extracts the 'Region' from an AWS EC2 instance metadata JSON response.\n\n    Args:\n        metadata_json (str): AWS EC2 instance metadata JSON response as a string.\n\n    Returns:\n        str: The 'Region' extracted from the metadata JSON response.\n    \"\"\"\n    try:\n        # Load the JSON response into a Python dictionary\n        metadata = json.loads(metadata_json)\n\n        # Extract the 'Region' from the metadata dictionary\n        region = metadata['placement']['availabilityZone'][:-1]\n\n        return region\n\n    except KeyError as e:\n        print(f\"Error: {e} not found in the metadata JSON response.\")\n        return None\n\n    except json.JSONDecodeError as e:\n 